# 05 · Survival Analysis
**Purpose:** Answer the question no binary churn model can: *At what month does a customer from each tier become profitable?*

The breakeven month per tier (from the CAC analysis) is overlaid on the Kaplan-Meier survival curve to compute the probability that a customer survives long enough to repay their acquisition cost. This is the single most financially consequential analysis in the project.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore'); np.random.seed(42)

cust = pd.read_csv('../data/processed/clean_saas_customers.csv')
cust['smo'] = pd.to_datetime(cust['signup_month'], errors='coerce')
cust['cdt'] = pd.to_datetime(cust['churn_date'],   errors='coerce')
ref = pd.Timestamp('2024-01-01')
cust['dur'] = np.where(cust['churned']==1,
    ((cust['cdt']-cust['smo'])/pd.Timedelta(days=30)).clip(0.5,60),
    ((ref-cust['smo'])/pd.Timedelta(days=30)).clip(0.5,60))
cust['dur'] = pd.to_numeric(cust['dur'],errors='coerce').fillna(6.0)
surv = cust[['tier','dur','churned','mrr']].dropna()
surv = surv[surv['dur']>0].copy()
print(surv.groupby('tier')[['dur','churned']].agg(['mean','count']).round(2))

In [ ]:
def km_curve(T, E, label, color, ax, breakeven_mo=None, breakeven_label=''):
    T,E = np.array(T,float), np.array(E,int)
    times = np.sort(np.unique(T[E==1])); S=1.0; curve_t=[0]; curve_s=[1.0]
    for t in times:
        ar=(T>=t).sum(); ev=((T==t)&(E==1)).sum()
        if ar>0: S*=(1-ev/ar)
        curve_t.append(t); curve_s.append(S)
    ax.step(curve_t, curve_s, where='post', label=label, color=color, linewidth=2.5)
    if breakeven_mo:
        # Find survival at breakeven
        Sq=1.0
        for t in times:
            if t>breakeven_mo: break
            ar=(T>=t).sum(); ev=((T==t)&(E==1)).sum()
            if ar>0: Sq*=(1-ev/ar)
        ax.axvline(x=breakeven_mo, color=color, linestyle='--', alpha=0.6, linewidth=1.5)
        ax.annotate(f'{breakeven_label}\nbe={breakeven_mo}mo\nS={Sq:.0%}',
            xy=(breakeven_mo, Sq), xytext=(breakeven_mo+1.5, Sq+0.07),
            fontsize=8, color=color, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=color, lw=1.2))
    return curve_t, curve_s

fig, ax = plt.subplots(figsize=(12,6))
colors = {'starter':'#e74c3c','pro':'#f39c12','enterprise':'#27ae60'}
breakevens = {'starter':(7.1,'Starter'),'pro':(4.1,'Pro'),'enterprise':(2.1,'Enterprise')}
for tier, color in colors.items():
    m = surv['tier']==tier
    km_curve(surv.loc[m,'dur'], surv.loc[m,'churned'], 
             f"{tier.title()} (n={m.sum()})", color, ax,
             breakeven_mo=breakevens[tier][0], breakeven_label=breakevens[tier][1])
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='50% survival threshold')
ax.fill_betweenx([0,1], 0, 7.1, alpha=0.04, color='red')
ax.set_xlabel('Months since signup', fontsize=12)
ax.set_ylabel('Survival probability S(t)', fontsize=12)
ax.set_title('Kaplan-Meier Survival Curves by Tier\n(dashed lines = CAC breakeven month)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.set_xlim(0,30); ax.set_ylim(0,1.05)
ax.set_facecolor('#f9f9f9'); fig.patch.set_facecolor('white')
plt.tight_layout(); plt.savefig('../reports/survival_km_breakeven.png', dpi=150); plt.show()
print("Red shaded area = months before Starter breakeven. Customers who churn in this window destroy capital.")

## Business Translation

The chart above contains the most actionable finding in this project:

- **Enterprise:** Breakeven at month 2.1 — survival rate at that point is ~98%. Almost every Enterprise customer recoups their acquisition cost. **Acquiring more Enterprise customers is unambiguously value-creating.**

- **Pro:** Breakeven at month 4.1 — survival rate ~83%. Most Pro customers are profitable. The 17% who churn early represent recoverable value if onboarding is improved.

- **Starter:** Breakeven at month 7.1 — survival rate **~58%**. **42% of every Starter cohort destroys capital.** The expected net value per Starter customer is −$126. Every dollar spent acquiring Starter customers has a negative expected return under current churn conditions.

**Implication:** Reducing Starter CAC, increasing Starter pricing, or improving Starter retention to month 7+ are all valid responses. Continuing to acquire Starter customers at scale without addressing this is the highest-priority strategic risk in the business.